# Problem Framing & Metrics

Translating ambiguous business asks into ML tasks is the most critical — and most often fumbled — part of an ML system design interview. This note covers the framing workflow, 6 worked examples, metric hierarchies, and a code implementation of key ranking metrics.

## What Interviewers Test
- Can you translate vague business goals to concrete ML predictions?
- Do you know the difference between north-star, proxy, and guardrail metrics?
- Can you explain why a metric might fail as a proxy (Goodhart's Law)?
- Do you know when to use NDCG vs MAP vs MRR?
- Can you implement a calibration curve and explain why calibration matters for auctions?

## Framing Workflow

```
Business ask → ML task type → What do you predict? → Label definition → Metric
```

**Task types:**
- **Binary classification:** fraud, click, conversion
- **Multi-class classification:** intent detection, content category
- **Regression:** bid price, time-to-purchase, satisfaction score
- **Ranking:** search results, feed ordering
- **Generation:** summarization, Q&A


## 6 Worked Framing Examples

| Business ask | ML task | What do you predict? | Label | Metric |
|---|---|---|---|---|
| "Increase engagement" | Ranking | P(user watches ≥ 30s) | 30-second view | NDCG on watch-time |
| "Reduce churn" | Binary classification | P(cancel within 30 days) | Churned in 30d | PR-AUC |
| "Show relevant ads" | CTR prediction | P(click given impression) | Click | Log-loss + calibration |
| "Surface safe content" | Binary classification | P(violates policy) | Human label | Precision@review queue |
| "Answer customer question" | Retrieval + generation | Relevant passage + answer | Human eval | ROUGE / LLM-judge |
| "Suggest next product" | Ranking | P(purchase given shown) | Purchase | NDCG@10, revenue@K |

> 💡 **Interview Tip:** Interviewers probe whether you distinguish the *proxy* (what you can measure offline) from the *true goal* (what the business cares about). For "increase engagement," the proxy is click-rate but the true goal is satisfaction — these can diverge.


## Metric Hierarchy

```
North-star metric (long-term business goal)
  ↑
Primary online metric (A/B test signal)
  ↑
Offline metric (proxy, pre-deployment)
  ↑
Loss function (what the model optimizes)
```

**Key principle:** Each level is a noisier/faster proxy for the level above. Metric selection is about choosing the best proxy at each level while understanding when they diverge.


In [ ]:
import numpy as np
from sklearn.metrics import ndcg_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)

# -------- NDCG --------
def dcg(r, k=None):
    r = np.array(r[:k], dtype=float)
    return (r / np.log2(np.arange(2, len(r)+2))).sum()

def ndcg(r, k=None):
    ideal = dcg(sorted(r, reverse=True), k)
    return 0.0 if ideal == 0 else dcg(r, k) / ideal

# -------- MAP --------
def average_precision(r):
    """r: binary relevance vector in ranked order."""
    r = np.array(r, dtype=float)
    prec_at_k = np.cumsum(r) / (np.arange(len(r)) + 1)
    return (prec_at_k * r).sum() / r.sum() if r.sum() > 0 else 0.0

def mean_average_precision(results):
    return np.mean([average_precision(r) for r in results])

# -------- MRR --------
def mean_reciprocal_rank(results):
    def rr(r):
        for i, rel in enumerate(r):
            if rel > 0:
                return 1 / (i + 1)
        return 0.0
    return np.mean([rr(r) for r in results])

# Synthetic ranking results (3 queries, ranked lists)
queries = [
    [3, 2, 0, 1, 0],   # Q1: good ranking
    [0, 0, 1, 2, 3],   # Q2: bad ranking (relevant at bottom)
    [1, 0, 1, 0, 1],   # Q3: binary, decent
]
binary = [[1 if x > 0 else 0 for x in q] for q in queries]

for i, (q, b) in enumerate(zip(queries, binary)):
    print(f"Q{i+1}: NDCG@5={ndcg(q,5):.3f}, AP={average_precision(b):.3f}, RR={1/(next(j+1 for j,x in enumerate(b) if x>0)):.3f}")

print(f"MAP: {mean_average_precision(binary):.3f}")
print(f"MRR: {mean_reciprocal_rank(binary):.3f}")


## Metric Comparison Table

| Metric | What it measures | Position sensitivity | Use when |
|---|---|---|---|
| **NDCG@K** | Ranked relevance, graded | Yes (top-heavy) | Search, recsys ranking |
| **MAP** | Precision across recall levels | Yes (uniform) | IR, document retrieval |
| **MRR** | Position of first relevant result | Yes (only first) | Q&A, one-answer queries |
| **Recall@K** | How many positives in top K | No | Retrieval candidate recall |
| **Precision@K** | Hit rate in top K | No | Trust & safety review queue |
| **AUC-ROC** | Threshold-free ranking | No | Binary classification |
| **PR-AUC** | Precision-recall tradeoff | No | Imbalanced classification |


## Calibration: Critical for Ads Auctions

A model with good AUC but poor calibration is dangerous in ad auctions — bids are based on predicted CTR, so miscalibration shifts money.


In [ ]:
from sklearn.calibration import calibration_curve

# Simulate an uncalibrated model (overconfident positives)
n = 2000
y_true = (np.random.rand(n) < 0.15).astype(int)  # 15% positive rate
# Overconfident: scores clustered near 0 and 1
y_scores_uncal = np.where(y_true == 1,
    np.clip(np.random.beta(5,1,n), 0.01, 0.99),
    np.clip(np.random.beta(1,5,n), 0.01, 0.99))
# Well-calibrated: scores match probabilities
y_scores_cal   = np.where(y_true == 1,
    np.clip(np.random.beta(3,2,n), 0.01, 0.99),
    np.clip(np.random.beta(1,6,n), 0.01, 0.99))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (scores, label) in zip(axes,
    [(y_scores_uncal, 'Uncalibrated'), (y_scores_cal, 'Calibrated')]):
    fop, mpv = calibration_curve(y_true, scores, n_bins=10)
    ax.plot(mpv, fop, 'b-o', label='Model')
    ax.plot([0,1],[0,1], 'r--', label='Perfect calibration')
    ax.set_title(f'{label} Model'); ax.set_xlabel('Mean predicted prob')
    ax.set_ylabel('Fraction of positives'); ax.legend()
plt.tight_layout()
plt.savefig('/tmp/calibration.png', dpi=80); plt.close()

# ECE (Expected Calibration Error)
def ece(y_true, y_score, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    ece_val = 0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_score >= lo) & (y_score < hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_score[mask].mean()
        ece_val += mask.mean() * abs(acc - conf)
    return ece_val

print(f"ECE uncalibrated: {ece(y_true, y_scores_uncal):.4f}")
print(f"ECE calibrated:   {ece(y_true, y_scores_cal):.4f}")
print("Lower ECE = better calibrated")


## Common Interview Questions

**Q: What is the proxy-metric pitfall (Goodhart's Law)?**
When a measure becomes a target, it ceases to be a good measure. Optimizing click-rate directly causes clickbait. Optimizing watch-time causes autoplaying junk. Always pair your primary metric with guardrails (e.g., maximize CTR subject to user satisfaction not decreasing). The way you detect this is divergence between offline (proxy) and online (true) metrics.

**Q: When would you choose MAP over NDCG?**
MAP uses binary relevance (relevant or not) and gives equal weight to all positions in the precision-recall curve. NDCG supports graded relevance (0, 1, 2, 3) and is top-heavy (earlier positions matter more). Use MAP for document retrieval where relevance is binary; use NDCG when you have rich relevance signals (engagement scores, dwell time).

**Q: Why does calibration matter for ad auctions?**
In second-price auctions, the optimal bid is CTR × value. If CTR predictions are systematically over- or under-estimated, bidders will shade their bids incorrectly, leading to revenue loss or welfare loss. Post-hoc calibration (Platt scaling, isotonic regression) is a standard step before plugging a model into an auction system.

**Q: What is Recall@K and when is it the right metric?**
Recall@K measures what fraction of all relevant items appear in the top-K retrieved. It's the right metric for the retrieval stage (candidate generation), where you want to ensure you don't miss relevant items — precision matters less at this stage and will be handled by the ranker.

## Key Takeaways
- Always translate business asks into: prediction task → label → metric
- Define north-star (long-term), proxy (offline), and guardrail metrics together
- Goodhart's Law: optimizing a proxy degrades the true goal; guardrails catch this
- NDCG: graded relevance, top-heavy; MAP: binary relevance, equal-weight; MRR: first hit
- Calibration = predicted probabilities match observed frequencies; critical for auctions
- PR-AUC is better than ROC-AUC for imbalanced classification